In [1]:
pip install gradio

Note: you may need to restart the kernel to use updated packages.


In [2]:
import gradio as gr
import numpy as np
import tensorflow as tf
import cv2

# ── Load your saved model ─────────────────────────────────────────────────────
model = tf.keras.models.load_model(r"C:\Users\deytu\Project - Medical Imaging\checkpoints\baseline_best.h5")

THRESHOLD = 0.10  # Your optimal threshold from evaluation

# ── Same preprocessing as training ───────────────────────────────────────────
def preprocess_xray_for_generator(img_array):
    img_gray = cv2.cvtColor(img_array.astype(np.uint8), cv2.COLOR_RGB2GRAY)
    clahe    = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img_clahe = clahe.apply(img_gray)
    img_norm  = img_clahe.astype(np.float32) / 255.0
    return np.stack([img_norm] * 3, axis=-1)

# ── Prediction function ───────────────────────────────────────────────────────
def predict_tb(image):
    if image is None:
        return "Please upload an image.", {}, ""

    # Resize and preprocess
    img_resized     = cv2.resize(image, (224, 224))
    img_preprocessed = preprocess_xray_for_generator(img_resized)
    img_input       = np.expand_dims(img_preprocessed, axis=0)

    # Predict
    prob_tb     = float(model.predict(img_input, verbose=0)[0][0])
    prob_normal = 1.0 - prob_tb
    prediction  = "TB Positive" if prob_tb > THRESHOLD else "Normal"

    # Result label
    label = f"{prediction} (score: {prob_tb:.3f}, threshold: {THRESHOLD})"

    # Probability dict for Gradio label component
    probs = {
        "TB Positive": prob_tb,
        "Normal":      prob_normal
    }

    # Clinical note
    if prob_tb > THRESHOLD:
        note = "Findings suggest possible TB. Recommend follow-up sputum test and clinical review."
    else:
        note = "No significant TB findings detected. Continue routine monitoring if symptomatic."

    disclaimer = "\n\nFor clinical reference only. Not a substitute for professional diagnosis."

    return label, probs, note + disclaimer

# ── Build the Gradio interface ────────────────────────────────────────────────
with gr.Blocks(title="TB Chest X-Ray Classifier") as demo:

    gr.Markdown("# TB Chest X-Ray Classifier")
    gr.Markdown("Upload a chest X-ray image to screen for Tuberculosis using a CNN trained on 4,200 images.")

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(
                type="numpy",
                label="Upload chest X-ray",
                height=300
            )
            submit_btn = gr.Button("Analyse X-ray", variant="primary")

        with gr.Column():
            result_label = gr.Label(
                label="Prediction",
                num_top_classes=2
            )
            result_text  = gr.Textbox(
                label="Clinical note",
                lines=4
            )
            raw_output   = gr.Textbox(
                label="Raw output",
                lines=1
            )

    gr.Markdown("---")
    gr.Markdown(
        f"**Model:** Baseline CNN  |  "
        f"**AUC:** 0.978  |  "
        f"**Sensitivity:** 89.6%  |  "
        f"**Specificity:** 98.9%  |  "
        f"**Threshold:** {THRESHOLD}"
    )

    submit_btn.click(
        fn=predict_tb,
        inputs=image_input,
        outputs=[raw_output, result_label, result_text]
    )

    # Example images from your test set (optional)
    gr.Examples(
        examples=[
            ['data/processed/test/TB/sample_tb.jpg'],
            ['data/processed/test/Normal/sample_normal.jpg'],
        ],
        inputs=image_input,
        label="Try example X-rays"
    )

demo.launch(share=True)  # share=True gives a public URL valid for 72 hours

* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
